## MSE / RMSE by model and test window

# Evaluation of Expanding Window Forecasts

Loads forecasts for 4 model types (`coherent`, `lc`, `hu`, `dl_country`) across 4 expanding-window train/test splits.

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "../data"

WINDOWS = [
    ("1959-1979", "1980-1989"),
    ("1959-1989", "1990-1999"),
    ("1959-1999", "2000-2009"),
    ("1959-2009", "2010-2019"),
]

SEEDS = [1, 2, 3]

# Column layout for the benchmark models (coherent/lc/hu):
#   country, sex, year, age, rate, variance, ?, ?, lower, upper
BENCHMARK_COLS = ["country", "sex", "year", "age", "rate",
                  "variance", "col6", "col7", "lower", "upper"]

# DL forecasts have 5 columns: country, sex, year, age, log_rate
DL_COLS = ["country", "sex", "year", "age", "log_rate"]

In [2]:
def load_benchmark(model, train, test, sep, ext):
    path = os.path.join(DATA_DIR, f"{model}_forecast_train{train}_test{test}.{ext}")
    df = pd.read_csv(path, sep=sep, header=None, names=BENCHMARK_COLS,
                     na_values=["NA"])
    df["model"] = model
    df["train"] = train
    df["test"] = test
    return df

def load_dl(train, test, seed):
    path = os.path.join(DATA_DIR,
        f"dl_country_forecast_train{train}_test{test}_seed{seed}.txt")
    df = pd.read_csv(path, sep=r"\s+", header=None, names=DL_COLS)
    df["seed"] = seed
    df["train"] = train
    df["test"] = test
    return df

In [17]:
# Load benchmark forecasts (coherent: .txt space-sep; lc/hu: .csv comma-sep)
coherent = pd.concat([load_benchmark("coherent", tr, te, sep=r"\s+", ext="txt")
                      for tr, te in WINDOWS], ignore_index=True)
lc = pd.concat([load_benchmark("lc", tr, te, sep=",", ext="csv")
                for tr, te in WINDOWS], ignore_index=True)
hu = pd.concat([load_benchmark("hu", tr, te, sep=",", ext="csv")
                for tr, te in WINDOWS], ignore_index=True)

print("coherent:", coherent.shape)
print("lc:      ", lc.shape)
print("hu:      ", hu.shape)
print("coherent['rate'].max():", coherent['rate'].max())
print(lc[lc['rate'] >= 100])
print("hu['rate'].max():", hu['rate'].max())

coherent: (314000, 13)
lc:       (314000, 13)
hu:       (314000, 13)
coherent['rate'].max(): 0.755791731536697
        country  sex  year  age        rate      variance  col6  col7  \
48899        76    0  1988   99  129.266806  2.277021e+35   NaN   NaN   
48999        76    0  1989   99  225.546316  1.556794e+38   NaN   NaN   
206799       76    0  2007   99  122.470709  5.539010e+19   NaN   NaN   
206899       76    0  2008   99  170.077911  1.710056e+21   NaN   NaN   
206999       76    0  2009   99  236.191136  4.850866e+22   NaN   NaN   
286699       76    0  2016   99  126.999093  1.805114e+12   NaN   NaN   
286799       76    0  2017   99  162.080223  1.562391e+13   NaN   NaN   
286899       76    0  2018   99  206.851860  1.265179e+14   NaN   NaN   
286999       76    0  2019   99  263.990827  9.703472e+14   NaN   NaN   

               lower         upper model      train       test  
48899   8.933146e-15  1.870551e+18    lc  1959-1979  1980-1989  
48999   1.040087e-15  4.8910

In [4]:
# Load DL forecasts (one file per seed); keep all seeds and also build the ensemble mean.
dl_all = pd.concat([load_dl(tr, te, s)
                    for tr, te in WINDOWS for s in SEEDS], ignore_index=True)

# Ensemble = mean log_rate across seeds, then exponentiate to a rate.
dl_ensemble = (dl_all
    .groupby(["country", "sex", "year", "age", "train", "test"], as_index=False)["log_rate"]
    .mean())
dl_ensemble["rate"] = np.exp(dl_ensemble["log_rate"])
dl_ensemble["model"] = "dl_country"

print("dl_all:     ", dl_all.shape)
print("dl_ensemble:", dl_ensemble.shape)
dl_ensemble.head()

dl_all:      (946800, 8)
dl_ensemble: (315600, 9)


,country,sex,year,age,train,test,log_rate,rate,model
0,50.0,0.0,1980.0,0.0,1959-1979,1980-1989,-4.552009,0.010546,dl_country
1,50.0,0.0,1980.0,1.0,1959-1979,1980-1989,-7.100457,0.000825,dl_country
2,50.0,0.0,1980.0,2.0,1959-1979,1980-1989,-7.594722,0.000503,dl_country
3,50.0,0.0,1980.0,3.0,1959-1979,1980-1989,-7.900791,0.000370,dl_country
4,50.0,0.0,1980.0,4.0,1959-1979,1980-1989,-8.044007,0.000321,dl_country


In [5]:
# Quick sanity check: rows per model per window
for name, df in [("coherent", coherent), ("lc", lc), ("hu", hu), ("dl_ensemble", dl_ensemble)]:
    print(name)
    print(df.groupby(["train", "test"]).size())
    print()

coherent
train      test     
1959-1979  1980-1989    74000
1959-1989  1990-1999    80000
1959-1999  2000-2009    80000
1959-2009  2010-2019    80000
dtype: int64

lc
train      test     
1959-1979  1980-1989    74000
1959-1989  1990-1999    80000
1959-1999  2000-2009    80000
1959-2009  2010-2019    80000
dtype: int64

hu
train      test     
1959-1979  1980-1989    74000
1959-1989  1990-1999    80000
1959-1999  2000-2009    80000
1959-2009  2010-2019    80000
dtype: int64

dl_ensemble
train      test     
1959-1979  1980-1989    78600
1959-1989  1990-1999    80000
1959-1999  2000-2009    80000
1959-2009  2010-2019    77000
dtype: int64



## Load actual test data

One `country_test_XXXX_XXXX.txt` file per test window. Same 5-column layout as the DL forecasts: country, sex, year, age, rate.

In [7]:
ACTUAL_COLS = ["country", "sex", "year", "age", "rate"]

def load_actual(test):
    # test is e.g. "1980-1989"; filename uses underscore: country_test_1980_1989.txt
    fname = f"country_test_{test.replace('-', '_')}.txt"
    df = pd.read_csv(os.path.join(DATA_DIR, fname),
                     sep=r"\s+", header=None, names=ACTUAL_COLS)
    df["test"] = test
    return df

actual = pd.concat([load_actual(te) for _, te in WINDOWS], ignore_index=True)
print("actual:", actual.shape)
print(actual.groupby("test").size())
actual.head()

actual: (315600, 6)
test
1980-1989    78600
1990-1999    80000
2000-2009    80000
2010-2019    77000
dtype: int64


,country,sex,year,age,rate,test
0,50.0,0.0,1980.0,0.0,0.009591,1980-1989
1,50.0,1.0,1980.0,0.0,0.011756,1980-1989
2,50.0,0.0,1980.0,1.0,0.000768,1980-1989
3,50.0,1.0,1980.0,1.0,0.001011,1980-1989
4,50.0,0.0,1980.0,2.0,0.000426,1980-1989


In [8]:
KEYS = ["country", "sex", "year", "age", "test"]

def errors_by_window(forecast_df, actual_df, model_name):
    merged = forecast_df[KEYS + ["rate"]].merge(
        actual_df[KEYS + ["rate"]],
        on=KEYS, suffixes=("_pred", "_true"))
    diff2 = (merged["rate_pred"] - merged["rate_true"]) ** 2
    out = (merged.assign(sq_err=diff2)
                 .groupby("test")["sq_err"]
                 .agg(["mean", "count"])
                 .rename(columns={"mean": "mse"}))
    out["rmse"] = np.sqrt(out["mse"])
    out["model"] = model_name
    return out.reset_index()

# Make sure key dtypes line up across forecast/actual frames before merging.
for df in (coherent, lc, hu, dl_ensemble, actual):
    for k in ["country", "sex", "year", "age"]:
        df[k] = df[k].astype(int)

results = pd.concat([
    errors_by_window(coherent,    actual, "coherent"),
    errors_by_window(lc,          actual, "lc"),
    errors_by_window(hu,          actual, "hu"),
    errors_by_window(dl_ensemble, actual, "dl_country"),
], ignore_index=True)

table = (results
    .pivot(index="test", columns="model", values=["mse", "rmse"])
    .swaplevel(axis=1)
    .sort_index(axis=1))
print(table)
table

model      coherent           dl_country                  hu            \
                mse      rmse        mse      rmse       mse      rmse   
test                                                                     
1980-1989  0.000865  0.029410   0.000840  0.028984  0.000964  0.031046   
1990-1999  0.000795  0.028203   0.000721  0.026850  0.001007  0.031725   
2000-2009  0.000453  0.021288   0.000315  0.017750  0.000628  0.025068   
2010-2019  0.000401  0.020033   0.000326  0.018058  0.004108  0.064093   

model            lc            
                mse      rmse  
test                           
1980-1989  1.020730  1.010312  
1990-1999  0.003497  0.059136  
2000-2009  1.443823  1.201592  
2010-2019  2.322120  1.523850  


model      coherent           dl_country                  hu            \
                mse      rmse        mse      rmse       mse      rmse   
test                                                                     
1980-1989  0.000865  0.029410   0.000840  0.028984  0.000964  0.031046   
1990-1999  0.000795  0.028203   0.000721  0.026850  0.001007  0.031725   
2000-2009  0.000453  0.021288   0.000315  0.017750  0.000628  0.025068   
2010-2019  0.000401  0.020033   0.000326  0.018058  0.004108  0.064093   

model            lc            
                mse      rmse  
test                           
1980-1989  1.020730  1.010312  
1990-1999  0.003497  0.059136  
2000-2009  1.443823  1.201592  
2010-2019  2.322120  1.523850

## MSE in log scale

Convert non-logged rate predictions to log scale; replace 0s with 9e-06 before taking the log.

In [19]:
FLOOR = 9e-06

def to_log(rate):
    return np.log(np.where(rate <= 0, FLOOR, rate))

def log_errors_by_window(forecast_df, actual_df, model_name, pred_col="rate", pred_is_log=False):
    merged = forecast_df[KEYS + [pred_col]].rename(columns={pred_col: "pred"}).merge(
        actual_df[KEYS + ["rate"]].rename(columns={"rate": "true"}),
        on=KEYS)
    log_pred = merged["pred"].values if pred_is_log else to_log(merged["pred"].values)
    log_true = to_log(merged["true"].values)
    diff2 = (log_pred - log_true) ** 2
    out = (merged.assign(sq_err=diff2)
                 .groupby("test")["sq_err"]
                 .agg(["mean", "count"])
                 .rename(columns={"mean": "log_mse"}))
    out["log_rmse"] = np.sqrt(out["log_mse"])
    out["model"] = model_name
    return out.reset_index()

log_results = pd.concat([
    log_errors_by_window(coherent,    actual, "coherent"),
    log_errors_by_window(lc,          actual, "lc"),
    log_errors_by_window(hu,          actual, "hu"),
    log_errors_by_window(dl_ensemble, actual, "dl_country",
                         pred_col="log_rate", pred_is_log=True),
], ignore_index=True)

log_table = (log_results
    .pivot(index="test", columns="model", values=["log_mse", "log_rmse"])
    .swaplevel(axis=1)
    .sort_index(axis=1))
print(log_table)
log_table

model      coherent           dl_country                  hu            \
            log_mse  log_rmse    log_mse  log_rmse   log_mse  log_rmse   
test                                                                     
1980-1989  0.175476  0.418899   0.169508  0.411714  0.186319  0.431647   
1990-1999  0.157505  0.396869   0.146268  0.382450  0.163490  0.404339   
2000-2009  0.167441  0.409196   0.151939  0.389793  0.189245  0.435023   
2010-2019  0.184291  0.429291   0.171249  0.413823  0.211098  0.459455   

model            lc            
            log_mse  log_rmse  
test                           
1980-1989  0.196456  0.443233  
1990-1999  0.188924  0.434654  
2000-2009  0.208679  0.456814  
2010-2019  0.241152  0.491072  


model      coherent           dl_country                  hu            \
            log_mse  log_rmse    log_mse  log_rmse   log_mse  log_rmse   
test                                                                     
1980-1989  0.175476  0.418899   0.169508  0.411714  0.186319  0.431647   
1990-1999  0.157505  0.396869   0.146268  0.382450  0.163490  0.404339   
2000-2009  0.167441  0.409196   0.151939  0.389793  0.189245  0.435023   
2010-2019  0.184291  0.429291   0.171249  0.413823  0.211098  0.459455   

model            lc            
            log_mse  log_rmse  
test                           
1980-1989  0.196456  0.443233  
1990-1999  0.188924  0.434654  
2000-2009  0.208679  0.456814  
2010-2019  0.241152  0.491072